# Cybersec OpenEnv: end-to-end RL train + eval (iter-3, remote-env)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LonelyGuy-SE1/cybersec_env/blob/main/notebooks/cybersec_grpo.ipynb)

One notebook, one Run-All. The full pipeline talks to the
[`cybersec` OpenEnv on Hugging Face Spaces](https://huggingface.co/spaces/Lonelyguyse1/cybersec)
over the OpenEnv WebSocket protocol -- the same way a hackathon judge
will. No vendored env state, no `localhost` shortcut.

1. Detect Colab / Kaggle / local and install dependencies.
2. Pull the `cybersec` package from the HF Space (needed only because
   the GRPO reward functions clone the env via `pickle` -- the network
   client isn't picklable). We then `pip install -e` it locally.
3. **Baseline** RandomPolicy + HeuristicPolicy over `MODE["n_baseline_episodes"]`
   seeds × 3 train scenarios + 1 held-out OOD scenario, all hitting the
   live HF Space.
4. Build a ~1500-prompt GRPO dataset by rolling the heuristic out
   **locally** (each row carries a pickled env snapshot for the trainer
   to clone and score candidate completions against).
5. Fine-tune **Qwen2.5-1.5B-Instruct** with **GRPO + Unsloth QLoRA** for
   `MODE["grpo_max_steps"]` steps × 4 generations/prompt. Eight reward
   functions imported from `cybersec.training.rewards`:
     * 4 schema/validity rewards
     * 1 actual env-step reward (clones a snapshot, applies the candidate)
     * 1 exfil-path shaping prior
     * 2 **anti-collapse rewards**: action-diversity within a group, and
       observation-aware (state-conditioned) behaviour
6. Re-run the trained adapter **over the live HF Space** on train +
   held-out scenarios.
7. Plot before/after reward curves, KL/entropy/component diagnostics.
8. Sanity-check: fail loud on iter-1-style mode collapse
   (`std_return == 0` across all training scenarios).

**Target compute**: free Colab T4 / Kaggle P100. Wall clock ≈ **55-75
min** at the defaults; baseline+eval over the network adds ~15 min vs
the iter-2 local-env defaults.


## 0. Detect runtime & install dependencies


In [ ]:
# This cell runs on plain Colab, Colab Pro, Kaggle kernels, or a local
# laptop. `RUNTIME` records which one. Artifacts go to a writable folder
# under whichever working directory exists.
import os, sys, subprocess
from pathlib import Path

if "google.colab" in sys.modules:
    RUNTIME = "colab"
    WORKDIR = Path("/content/cybersec_workdir"); WORKDIR.mkdir(exist_ok=True)
elif Path("/kaggle/working").exists() or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    RUNTIME = "kaggle"
    WORKDIR = Path("/kaggle/working/cybersec_workdir"); WORKDIR.mkdir(exist_ok=True)
else:
    RUNTIME = "local"
    WORKDIR = Path.cwd() / "_artifacts"; WORKDIR.mkdir(exist_ok=True)

print(f"runtime: {RUNTIME}")
print(f"workdir: {WORKDIR}")


In [ ]:
%%capture
# Pull the env package straight from the HF Space (the canonical source)
# so this notebook is self-contained -- no GitHub clone required.
#
# WHY we install the package locally even though baselines + eval go
# over the network: the GRPO reward function `reward_step_total` clones
# the env via pickle.dumps/loads to roll candidate completions forward
# and grade them. The OpenEnv WebSocket client is intentionally NOT
# picklable (it owns a live socket), so the trainer has to keep a local
# copy of the env to clone. Baseline + post-train eval still hit the
# remote API.

HF_SPACE_REPO_ID = "Lonelyguyse1/cybersec"
HF_SPACE_REVISION = "main"
PKG_STAGE = WORKDIR / "cybersec_env_pkg"

if RUNTIME in {"colab", "kaggle"}:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub>=0.24", "openenv-core>=0.2.2"], check=True)
    from huggingface_hub import snapshot_download
    PKG_STAGE.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_SPACE_REPO_ID,
        repo_type="space",
        revision=HF_SPACE_REVISION,
        local_dir=str(PKG_STAGE),
        local_dir_use_symlinks=False,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-e", str(PKG_STAGE)], check=True)
else:
    print(f"local runtime; skipping snapshot_download (assumes pip install -e .)")

# GRPO stack. Unsloth pulls torch + bitsandbytes; install it first so its
# pinned wheels win.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "trl", "peft", "accelerate", "bitsandbytes"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "pandas"], check=False)


## 1. Imports & config


In [ ]:
from __future__ import annotations

# Iter-1's notebook output was buried under thousands of identical
# transformers warnings. Silence the known noise *before* importing
# torch / transformers so the cell outputs are readable.
import warnings, logging
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("accelerate").setLevel(logging.ERROR)
logging.getLogger("peft").setLevel(logging.ERROR)
logging.getLogger("websockets").setLevel(logging.WARNING)

import json
import math
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional

import matplotlib
if RUNTIME != "colab":
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset

from cybersec import (
    ActionType,
    CybersecAction,
    CybersecEnv,
    CybersecEnvironment,
    CybersecObservation,
    list_eval_scenarios,
    list_scenarios,
    list_train_scenarios,
)
from cybersec.baselines import (
    EpisodeResult,
    HeuristicPolicy,
    RandomPolicy,
    aggregate_results,
    run_episode,
)
from cybersec.training.rewards import (
    SYSTEM_PROMPT,
    default_reward_funcs,
    parse_first_json_object,
    parsed_action,
    render_observation,
    snapshot_env,
)

# ---------------------------------------------------------------------------
# Single source of truth for every dial in this notebook.
# Defaults are tuned for a free Colab T4 / Kaggle P100 budget:
#
#   - 30-episode baseline over the network (4 scenarios):  ~12 min
#   - 1500-prompt dataset build (LOCAL):                    ~2 min
#   - Qwen 1.5B + Unsloth load (4-bit):                     ~2 min
#   - 100 GRPO steps x 4 generations:                       ~25 min
#   - 30-episode trained-policy eval over the network:      ~10 min
#   - 30-episode held-out OOD eval over the network:        ~3 min
#   - Plots, tables, persistence:                           ~1 min
#   ----------------------------------------------------------
#   total wall clock:                                       ~55 min
#
# `n_baseline_episodes`/`n_post_train_episodes` dominate network time;
# bump them to 50 if you have headroom and want tighter error bars.
# ---------------------------------------------------------------------------
MODE = {
    "remote_env_url":         "https://lonelyguyse1-cybersec.hf.space",
    "n_baseline_episodes":    30,
    "n_dataset_seeds":        30,
    "max_dataset_rows":       1500,
    "grpo_max_steps":         100,
    "grpo_num_generations":   4,
    "grpo_per_device_bs":     2,
    "grpo_grad_accum":        4,
    "grpo_logging_steps":     5,
    "grpo_save_steps":        50,
    "n_post_train_episodes":  30,
    # Set to "local" to develop offline against the in-process env
    # (faster, but doesn't exercise the OpenEnv WebSocket protocol).
    # Default "remote" hits the deployed HF Space, the way judges will.
    "eval_target":            "remote",
}

ARTIFACTS = WORKDIR

ADAPTER_DIR        = ARTIFACTS / "qwen_cybersec_lora"
TRAIN_LOG          = ARTIFACTS / "training_log.json"
BASELINE_JSON      = ARTIFACTS / "baseline_metrics.json"
POST_JSON          = ARTIFACTS / "post_train_metrics.json"
HELDOUT_JSON       = ARTIFACTS / "heldout_metrics.json"
BASELINE_PLOT      = ARTIFACTS / "baseline_curves.png"
BEFORE_AFTER       = ARTIFACTS / "before_after_curves.png"
TRAIN_DIAGNOSTICS  = ARTIFACTS / "training_diagnostics.png"
SUMMARY_MD         = ARTIFACTS / "summary_table.md"
TRAJECTORY_JSON    = ARTIFACTS / "trajectory_dataset.jsonl"

MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_PROMPT_LEN  = 1024
MAX_NEW_TOKENS  = 48

TRAIN_SCENARIOS = list_train_scenarios()
HELDOUT_SCENARIOS = list_eval_scenarios()
ALL_SCENARIOS = list_scenarios()
SEEDS_BASELINE = list(range(MODE["n_baseline_episodes"]))
SEEDS_DATASET  = list(range(MODE["n_dataset_seeds"]))
SEEDS_POST     = list(range(MODE["n_post_train_episodes"]))

print("train scenarios:   ", TRAIN_SCENARIOS)
print("held-out scenarios:", HELDOUT_SCENARIOS)
print("MODE:")
print(json.dumps(MODE, indent=2))


## 2. Verify the live HF Space is reachable

Before we run hundreds of episodes against it, make sure the Space is
healthy and exposes the schema we expect (6 actions, our scenarios).
This fails fast if the Space is asleep or the build broke.


In [ ]:
import urllib.request

def probe_space(base_url: str) -> dict:
    out = {}
    with urllib.request.urlopen(base_url + "/health", timeout=20) as r:
        out["health"] = json.loads(r.read())
    with urllib.request.urlopen(base_url + "/schema", timeout=20) as r:
        out["schema"] = json.loads(r.read())
    return out

probe = probe_space(MODE["remote_env_url"])
action_enum = (
    probe["schema"].get("action", {}).get("$defs", {})
    .get("ActionType", {}).get("enum", [])
)
print("space health:", probe["health"])
print("remote action enum:", action_enum)
assert "MONITOR" in action_enum and "ISOLATE_ASSET" in action_enum, (
    "remote action surface doesn't match this notebook -- did the Space "
    "rebuild against an older revision?"
)
print("OK: space is healthy and on the iter-3 action surface")


## 3. Run baselines against the deployed HF Space

We open one persistent WebSocket session per episode to the Space and
let `RandomPolicy` / `HeuristicPolicy` drive it through to terminal.
Same code that hits the local env -- only the env factory changes.


In [ ]:
def _open_remote_episode(client, seed, scenario_id):
    \"\"\"Reset a SyncEnvClient and return (observation, last_step_result).\"\"\"
    res = client.reset(seed=seed, scenario_id=scenario_id)
    return res

def run_episode_remote(
    client_factory,
    policy,
    seed,
    scenario_id,
    attacker_personality=None,
):
    \"\"\"Run one episode over the OpenEnv WebSocket protocol.

    `client_factory` is a callable returning a fresh SyncEnvClient (each
    episode opens its own WS session so seeds don't bleed). Returns the
    same `EpisodeResult` shape as `cybersec.baselines.run_episode`.
    \"\"\"
    if hasattr(policy, "reset"):
        policy.reset()

    invalid = 0
    fp = 0
    reward_curve = []
    final_obs = None
    final_info = {}

    client = client_factory()
    with client:
        kwargs = {"seed": seed, "scenario_id": scenario_id}
        if attacker_personality is not None:
            kwargs["attacker_personality"] = attacker_personality
        res = client.reset(**kwargs)
        obs = res.observation

        while not (res.done or obs.done):
            action = policy.act(obs)
            res = client.step(action)
            obs = res.observation
            rb = obs.info.get("reward_breakdown", {}) if isinstance(obs.info, dict) else {}
            if rb.get("invalid_action_penalty", 0.0):
                invalid += 1
            if rb.get("false_positive_penalty", 0.0):
                fp += 1
            reward_curve.append(float(obs.reward or 0.0))

        final_obs = obs
        final_info = obs.info if isinstance(obs.info, dict) else {}

    terminal = final_info.get("terminal", {}) if isinstance(final_info, dict) else {}
    return EpisodeResult(
        policy_name=policy.name,
        scenario_id=final_obs.scenario_id,
        attacker_personality=final_obs.attacker_personality.value
            if hasattr(final_obs.attacker_personality, "value")
            else str(final_obs.attacker_personality),
        seed=seed,
        steps=final_obs.tick,
        cumulative_reward=float(terminal.get("cumulative_reward", sum(reward_curve))),
        succeeded_stage_count=int(terminal.get("stages_succeeded", 0)),
        total_stage_count=int(terminal.get("stages_total", 0)),
        exfil_completed=bool(terminal.get("exfil_completed", False)),
        terminal_reason=terminal.get("terminal_reason"),
        confirmed_compromised=list(final_obs.confirmed_compromised),
        invalid_action_count=invalid,
        false_positive_count=fp,
        reward_curve=reward_curve,
    )


def make_eval_runner():
    \"\"\"Return a `run_episode`-shaped callable that uses MODE['eval_target'].\"\"\"
    if MODE["eval_target"] == "remote":
        base_url = MODE["remote_env_url"]
        def remote_factory():
            return CybersecEnv(base_url=base_url).sync()
        return lambda policy, seed, scenario_id: run_episode_remote(
            remote_factory, policy, seed, scenario_id
        )
    else:
        local_env = CybersecEnvironment()
        return lambda policy, seed, scenario_id: run_episode(
            local_env, policy, seed=seed, scenario_id=scenario_id
        )

eval_runner = make_eval_runner()
print("eval target:", MODE["eval_target"])


In [ ]:
baseline_runs = {}
t0 = time.time()
for sid in ALL_SCENARIOS:
    rand = [eval_runner(RandomPolicy(seed=s), seed=s, scenario_id=sid) for s in SEEDS_BASELINE]
    heur = [eval_runner(HeuristicPolicy(),    seed=s, scenario_id=sid) for s in SEEDS_BASELINE]
    baseline_runs[sid] = {"random": rand, "heuristic": heur}
    print(f"{sid:<32s}  random={aggregate_results(rand)['mean_return']:7.3f}"
          f"  heuristic={aggregate_results(heur)['mean_return']:7.3f}  "
          f"({MODE['n_baseline_episodes']} ep each)")
print(f"baseline elapsed: {time.time()-t0:.1f}s  ({MODE['eval_target']} env)")


### 3a. Persist baseline metrics + plot


In [ ]:
def _padded_cumulative(curves, target_len):
    out = np.zeros((len(curves), target_len), dtype=float)
    for i, c in enumerate(curves):
        out[i, : len(c)] = c
        if len(c) < target_len:
            out[i, len(c):] = c[-1] if c else 0.0
    return np.cumsum(out, axis=1)

n_panels = len(ALL_SCENARIOS)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4), sharey=True)
if n_panels == 1:
    axes = [axes]
for ax, sid in zip(axes, ALL_SCENARIOS):
    cell = baseline_runs[sid]
    horizon = max(max(len(r.reward_curve) for r in cell['random']),
                  max(len(r.reward_curve) for r in cell['heuristic']))
    for label, color in [("random", "tab:gray"), ("heuristic", "tab:blue")]:
        cumr = _padded_cumulative([r.reward_curve for r in cell[label]], horizon)
        mean = cumr.mean(axis=0); std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    held_out_tag = "  (HELD-OUT)" if sid in HELDOUT_SCENARIOS else ""
    ax.set_title(sid + held_out_tag, fontsize=10); ax.set_xlabel("tick"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward"); axes[0].legend()
fig.suptitle(
    f"Cybersec OpenEnv baselines (n={MODE['n_baseline_episodes']}/scenario, "
    f"env={MODE['eval_target']})"
)
fig.tight_layout(); fig.savefig(BASELINE_PLOT, dpi=140); plt.show()

baseline_metrics = {
    sid: {p: aggregate_results(cell[p]) for p in ("random", "heuristic")}
    for sid, cell in baseline_runs.items()
}
baseline_metrics["_meta"] = {
    "n_episodes": MODE["n_baseline_episodes"],
    "eval_target": MODE["eval_target"],
    "remote_env_url": MODE["remote_env_url"],
    "train_scenarios": TRAIN_SCENARIOS,
    "heldout_scenarios": HELDOUT_SCENARIOS,
}
BASELINE_JSON.write_text(json.dumps(baseline_metrics, indent=2))
print("wrote", BASELINE_JSON)


## 4. Build the GRPO training dataset (LOCAL)

This step **runs locally** on purpose. `reward_step_total` clones the
env via `pickle` to score every candidate completion the policy
generates, and the OpenEnv WebSocket client owns a live socket so it
isn't picklable. So we keep a `CybersecEnvironment` in-process for
training, then go back to the remote env for evaluation.

The held-out scenario (`cloud_metadata_ssrf`) is **deliberately not in
the training data** -- it exists only for the OOD post-training eval.


In [ ]:
local_env = CybersecEnvironment()
t0 = time.time()
rows = []
trajectory_lines = []
for sid in TRAIN_SCENARIOS:
    for seed in SEEDS_DATASET:
        ep_env = CybersecEnvironment()
        policy = HeuristicPolicy()
        obs = ep_env.reset(seed=seed, scenario_id=sid)
        while not obs.done:
            blob = snapshot_env(ep_env)
            prompt = render_observation(obs)
            rows.append({
                "prompt": prompt,
                "system": SYSTEM_PROMPT,
                "valid_assets": obs.valid_targets["assets"],
                "valid_identities": obs.valid_targets["identities"],
                "isolated_assets": list(obs.isolated_assets),
                "revoked_identities": list(obs.revoked_identities),
                "blocked_egress": list(obs.blocked_egress_assets),
                "patched": list(obs.patched_assets),
                "alert_count": len(obs.alerts),
                "env_snapshot": blob,
            })
            act = policy.act(obs)
            trajectory_lines.append(
                json.dumps({"prompt": prompt, "completion": act.model_dump_json()})
            )
            obs = ep_env.step(act)

random.Random(0).shuffle(rows)
rows = rows[: MODE["max_dataset_rows"]]
ds = Dataset.from_list(rows)
TRAJECTORY_JSON.write_text("\n".join(trajectory_lines))
print(f"dataset size: {len(ds)}  (train scenarios only, local env)  build elapsed: {time.time()-t0:.1f}s")
print(ds[0]["prompt"][:300])


## 5. Reward functions (eight independent signals)

All reward functions live in `cybersec.training.rewards` so the
notebook, the tests, and any future training script use the same
definitions.


In [ ]:
REWARD_FUNCS = default_reward_funcs()
REWARD_NAMES = [f.__name__ for f in REWARD_FUNCS]
print("reward components:")
for n in REWARD_NAMES:
    print(f"  - {n}")


## 6. Load Qwen2.5-1.5B with Unsloth (4-bit QLoRA)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_PROMPT_LEN + MAX_NEW_TOKENS,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=0,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model.print_trainable_parameters()


## 7. Format dataset with the chat template


In [ ]:
def to_chat_prompt(row):
    msgs = [
        {"role": "system", "content": row["system"]},
        {"role": "user",   "content": row["prompt"]},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return {"prompt": text}

ds_chat = ds.map(to_chat_prompt)
print("first prompt (truncated):")
print(ds_chat[0]["prompt"][:500])


## 8. GRPO trainer


In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(ARTIFACTS / "grpo_checkpoints"),
    learning_rate=5e-6,
    per_device_train_batch_size=MODE["grpo_per_device_bs"],
    gradient_accumulation_steps=MODE["grpo_grad_accum"],
    max_prompt_length=MAX_PROMPT_LEN,
    max_completion_length=MAX_NEW_TOKENS,
    num_generations=MODE["grpo_num_generations"],
    num_train_epochs=1,
    max_steps=MODE["grpo_max_steps"],
    logging_steps=MODE["grpo_logging_steps"],
    save_steps=MODE["grpo_save_steps"],
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    report_to=[],
    seed=0,
    # Mild sampling temperature so candidates within a group can diverge,
    # giving reward_action_diversity something to score.
    temperature=1.0,
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    train_dataset=ds_chat,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
)

t0 = time.time()
trainer.train()
print(f"GRPO train elapsed: {time.time()-t0:.1f}s")


## 9. Save adapter + training log


In [ ]:
ADAPTER_DIR.mkdir(exist_ok=True)
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

history = getattr(trainer.state, "log_history", []) or []
TRAIN_LOG.write_text(json.dumps({
    "reward_names": REWARD_NAMES,
    "history": history,
    "model_name": MODEL_NAME,
    "mode": MODE,
}, indent=2))
print("saved adapter to", ADAPTER_DIR)
print("wrote", TRAIN_LOG, "with", len(history), "log rows")


### 9a. Training diagnostics: KL, loss, per-component reward

These are the early-warning system: if KL is rising fast and
`reward_action_diversity` plateaus near `1/num_generations`, the policy
is mode-collapsing -- stop and adjust before the trained-vs-baseline
table flatters you with a memorised plan.


In [ ]:
log = json.loads(TRAIN_LOG.read_text()) if TRAIN_LOG.exists() else {"history": []}
hist = log.get("history") or []

if not hist:
    print("no training log rows -- did GRPO train?")
else:
    df_log = pd.DataFrame(hist)
    if "step" not in df_log.columns:
        df_log["step"] = range(len(df_log))

    component_cols = [c for c in df_log.columns
                      if c.startswith("rewards/") or c in REWARD_NAMES
                      or any(c.endswith(name) for name in REWARD_NAMES)]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    for col, color in [("loss", "tab:blue"), ("kl", "tab:red")]:
        if col in df_log.columns and df_log[col].notna().any():
            axes[0].plot(df_log["step"], df_log[col], label=col, color=color)
    axes[0].set_title("loss / KL"); axes[0].set_xlabel("step"); axes[0].legend()

    if "reward" in df_log.columns and df_log["reward"].notna().any():
        axes[1].plot(df_log["step"], df_log["reward"], label="reward", color="tab:green")
    if "completion_length" in df_log.columns and df_log["completion_length"].notna().any():
        ax2 = axes[1].twinx()
        ax2.plot(df_log["step"], df_log["completion_length"], label="completion_length",
                 color="tab:purple", linestyle="--")
        ax2.set_ylabel("completion_length", color="tab:purple")
    axes[1].set_title("total reward + completion length"); axes[1].set_xlabel("step")
    axes[1].legend(loc="upper left")

    if component_cols:
        for col in component_cols:
            if df_log[col].notna().any():
                short = col.split("/")[-1].replace("reward_", "")
                axes[2].plot(df_log["step"], df_log[col], label=short, alpha=0.85)
        axes[2].set_title("reward components per step")
        axes[2].set_xlabel("step")
        axes[2].legend(fontsize=7, ncol=2, loc="best")
    else:
        axes[2].set_title("(no per-component reward columns in log)")

    fig.tight_layout(); fig.savefig(TRAIN_DIAGNOSTICS, dpi=140); plt.show()
    print("wrote", TRAIN_DIAGNOSTICS)


## 10. Evaluate the trained policy AGAINST the live HF Space

Reload through **Unsloth** (not vanilla `transformers + peft`) on
purpose: GRPO trained the model with Unsloth's fused-QKV attention
patch, and the saved LoRA expects to plug into it. Then we wire the
trained policy through `eval_runner`, which dispatches to the remote
HF Space when `MODE['eval_target'] == 'remote'`.


In [ ]:
from unsloth import FastLanguageModel

# Free training-only buffers if the kernel still has them.
for _name in ("model", "trainer"):
    if _name in globals():
        del globals()[_name]
torch.cuda.empty_cache()

eval_model, eval_tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),
    max_seq_length=MAX_PROMPT_LEN + MAX_NEW_TOKENS,
    dtype=None,
    load_in_4bit=True,
)
eval_tokenizer.pad_token = eval_tokenizer.pad_token or eval_tokenizer.eos_token
FastLanguageModel.for_inference(eval_model)
print("loaded trained adapter via Unsloth for evaluation")


In [ ]:
@torch.inference_mode()
def llm_act(obs):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": render_observation(obs)},
    ]
    text = eval_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(text, return_tensors="pt").to(eval_model.device)
    out = eval_model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=eval_tokenizer.eos_token_id,
    )
    completion = eval_tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    a = parsed_action(completion)
    return a or CybersecAction(action_type=ActionType.MONITOR)

class TrainedLLMPolicy:
    name = "qwen-1.5b-grpo"
    def reset(self):
        pass
    def act(self, obs):
        return llm_act(obs)

# Re-derive the eval runner so the trained policy goes over the same
# transport (remote/local) as the baseline. Reuses the shared session
# pattern; each episode opens its own WebSocket.
eval_runner = make_eval_runner()

trained_runs = {}
t0 = time.time()
for sid in TRAIN_SCENARIOS:
    runs = [eval_runner(TrainedLLMPolicy(), seed=s, scenario_id=sid) for s in SEEDS_POST]
    trained_runs[sid] = runs
    agg = aggregate_results(runs)
    print(f"{sid:<32s}  trained-llm={agg['mean_return']:7.3f}  "
          f"std={float(np.std([r.cumulative_reward for r in runs])):.3f}")
print(f"trained-policy eval (train scenarios, env={MODE['eval_target']}) elapsed: {time.time()-t0:.1f}s")


### 10a. Held-out OOD evaluation

The whole point of holding `cloud_metadata_ssrf` out of training is
this cell: a clean read on whether the trained policy *generalises* or
just memorised per-scenario plans. Same transport, same code path.


In [ ]:
heldout_runs = {}
t0 = time.time()
for sid in HELDOUT_SCENARIOS:
    runs = [eval_runner(TrainedLLMPolicy(), seed=s, scenario_id=sid) for s in SEEDS_POST]
    heldout_runs[sid] = runs
    agg = aggregate_results(runs)
    rand_mean = aggregate_results(baseline_runs[sid]['random'])['mean_return']
    heur_mean = aggregate_results(baseline_runs[sid]['heuristic'])['mean_return']
    std = float(np.std([r.cumulative_reward for r in runs]))
    print(f"{sid:<32s}  trained-llm={agg['mean_return']:7.3f}  std={std:.3f}  "
          f"vs heur={heur_mean:.3f}  vs rand={rand_mean:.3f}")
print(f"trained-policy eval (held-out, env={MODE['eval_target']}) elapsed: {time.time()-t0:.1f}s")

heldout_metrics = {
    sid: {
        "random":      aggregate_results(baseline_runs[sid]["random"]),
        "heuristic":   aggregate_results(baseline_runs[sid]["heuristic"]),
        "trained_llm": aggregate_results(heldout_runs[sid]),
    }
    for sid in HELDOUT_SCENARIOS
}
HELDOUT_JSON.write_text(json.dumps(heldout_metrics, indent=2))
print("wrote", HELDOUT_JSON)


## 11. Before/after curves on identical axes


In [ ]:
all_post = {**trained_runs, **heldout_runs}
panels = ALL_SCENARIOS
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4), sharey=True)
if len(panels) == 1:
    axes = [axes]
for ax, sid in zip(axes, panels):
    cell = baseline_runs[sid]
    llm  = all_post.get(sid, [])
    horizon = max(
        max(len(r.reward_curve) for r in cell['random']),
        max(len(r.reward_curve) for r in cell['heuristic']),
        max((len(r.reward_curve) for r in llm), default=1),
    )
    palette = [("random", cell['random'], "tab:gray"),
               ("heuristic", cell['heuristic'], "tab:blue"),
               ("trained-llm", llm, "tab:red")]
    for label, runs, color in palette:
        if not runs:
            continue
        cumr = _padded_cumulative([r.reward_curve for r in runs], horizon)
        mean = cumr.mean(axis=0); std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    held_out_tag = "  (HELD-OUT)" if sid in HELDOUT_SCENARIOS else ""
    ax.set_title(sid + held_out_tag, fontsize=10); ax.set_xlabel("tick"); ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward"); axes[0].legend()
fig.suptitle(
    f"Cybersec OpenEnv -- pre/post GRPO ({MODE['grpo_max_steps']} steps), "
    f"n={MODE['n_post_train_episodes']}/scenario, env={MODE['eval_target']}"
)
fig.tight_layout(); fig.savefig(BEFORE_AFTER, dpi=140); plt.show()


## 12. Summary table + headline delta


In [ ]:
rows = []
for sid in ALL_SCENARIOS:
    cell = baseline_runs[sid]
    post_runs = all_post.get(sid, [])
    for policy_name, runs in [
        ("random",     cell["random"]),
        ("heuristic",  cell["heuristic"]),
        ("trained-llm", post_runs),
    ]:
        if not runs:
            continue
        agg = aggregate_results(runs)
        returns = [r.cumulative_reward for r in runs]
        rows.append({
            "scenario":     sid,
            "split":        "held-out" if sid in HELDOUT_SCENARIOS else "train",
            "policy":       policy_name,
            "mean_return":  agg["mean_return"],
            "std_return":   round(float(np.std(returns)), 3) if returns else 0.0,
            "mean_stages":  agg["mean_stages_succeeded"],
            "exfil_rate":   agg["exfil_rate"],
            "invalid_rate": agg["mean_invalid_actions"],
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

post_metrics = {
    sid: {
        "random":      aggregate_results(baseline_runs[sid]["random"]),
        "heuristic":   aggregate_results(baseline_runs[sid]["heuristic"]),
        "trained_llm": aggregate_results(all_post.get(sid, [])) if all_post.get(sid) else None,
    }
    for sid in ALL_SCENARIOS
}
post_metrics["_meta"] = {
    "n_post_episodes": MODE["n_post_train_episodes"],
    "grpo_max_steps":  MODE["grpo_max_steps"],
    "eval_target":     MODE["eval_target"],
    "remote_env_url":  MODE["remote_env_url"],
    "model":           MODEL_NAME,
    "adapter":         str(ADAPTER_DIR),
    "train_scenarios": TRAIN_SCENARIOS,
    "heldout_scenarios": HELDOUT_SCENARIOS,
    "reward_funcs":    REWARD_NAMES,
}
POST_JSON.write_text(json.dumps(post_metrics, indent=2))
SUMMARY_MD.write_text(df.to_markdown(index=False))
print("wrote", POST_JSON)
print("wrote", SUMMARY_MD)


In [ ]:
print("=== Headline delta (trained-llm vs heuristic) ===")
for sid in ALL_SCENARIOS:
    h = aggregate_results(baseline_runs[sid]["heuristic"])["mean_return"]
    runs = all_post.get(sid, [])
    if not runs:
        continue
    t = aggregate_results(runs)["mean_return"]
    tag = "(HELD-OUT)" if sid in HELDOUT_SCENARIOS else "(train)"
    print(f"{sid:<32s} {tag:<11s} heuristic={h:7.3f}  trained-llm={t:7.3f}  delta={t-h:+.3f}")


## 13. Sanity assertions (incl. iter-1 reward-hack canaries)

These are the gate before we ship a run as "good". The earlier checks
(invalid-action rate, divergence-vs-random) survive iter-1; the
**std_return** check is new in iter-2 -- it specifically catches the
mode-collapse failure where every seed produces the same trajectory.


In [ ]:
# 1. Reward shaping is healthy: heuristic must out-earn random *on aggregate*.
heur_total = sum(
    float(np.mean([r.cumulative_reward for r in cell['heuristic']]))
    for cell in baseline_runs.values()
)
rand_total = sum(
    float(np.mean([r.cumulative_reward for r in cell['random']]))
    for cell in baseline_runs.values()
)
print(f"baseline totals: heuristic={heur_total:+.3f}  random={rand_total:+.3f}")
assert heur_total > rand_total, (
    f"reward shaping regressed: heuristic total ({heur_total:.3f}) "
    f"<= random total ({rand_total:.3f}) summed across scenarios"
)

# 2. Trained policy must produce mostly-valid actions across all post-train
#    episodes. Aggregate over train + held-out to keep one number.
total_steps = sum(len(r.reward_curve) for runs in all_post.values() for r in runs)
total_invalid = sum(r.invalid_action_count for runs in all_post.values() for r in runs)
valid_rate = 1.0 - (total_invalid / max(1, total_steps))
print(f"valid-action rate across all post-train episodes: {valid_rate:.1%}  "
      f"({total_invalid} invalid / {total_steps} steps)")
assert valid_rate >= 0.8, (
    f"trained policy is producing too many invalid actions ({valid_rate:.1%} valid)"
)

# 3. Trained policy must not be catastrophically worse than random on any
#    *training* scenario. (Held-out scenarios are allowed to be worse;
#    we only print those.)
for sid in TRAIN_SCENARIOS:
    rand = float(np.mean([r.cumulative_reward for r in baseline_runs[sid]['random']]))
    llm  = float(np.mean([r.cumulative_reward for r in trained_runs[sid]]))
    assert llm >= rand - 5.0, (
        f"{sid}: trained-llm ({llm:.3f}) is more than 5 points below random "
        f"({rand:.3f}) -- training likely diverged"
    )

# 4. Reward-hack canary: the trained policy MUST have non-trivial variance on
#    at least one training scenario. std_return == 0.0 across all training
#    scenarios is iter-1's exact failure mode -- a memorised canned plan
#    that ignores the observation.
non_zero_std_scenarios = []
for sid in TRAIN_SCENARIOS:
    returns = [r.cumulative_reward for r in trained_runs[sid]]
    s = float(np.std(returns))
    print(f"{sid:<32s}  trained-llm std_return={s:.4f}")
    if s > 0.1:
        non_zero_std_scenarios.append(sid)
assert non_zero_std_scenarios, (
    "REWARD HACK CANARY: trained-llm has std_return == 0 on every training "
    "scenario -- this is exactly the iter-1 mode-collapse pattern. "
    "Inspect the training_diagnostics.png and the action distribution "
    "before publishing this run."
)

print("all sanity checks passed")


## 14. (Optional) Push artifacts back to the HF Space

Set `PUSH_ARTIFACTS = True` after a successful run if you want the
trained adapter and updated metrics to live on the HF Space alongside
the env (so judges can diff `_artifacts/` between iterations). Uses the
same `huggingface_hub` API as the install cell.


In [ ]:
PUSH_ARTIFACTS = False

if PUSH_ARTIFACTS:
    from huggingface_hub import HfApi
    api = HfApi()
    api.upload_folder(
        folder_path=str(ARTIFACTS),
        path_in_repo="_artifacts",
        repo_id=HF_SPACE_REPO_ID,
        repo_type="space",
        commit_message=f"iter-3 training artifacts (eval_target={MODE['eval_target']})",
    )
    print("uploaded artifacts to", HF_SPACE_REPO_ID)
else:
    print("artifact push skipped (set PUSH_ARTIFACTS=True to enable)")
